# Pipeline de Treinamento — Previsão de `geek_rating` (BoardGameGeek)

Este notebook treina e compara 4 modelos para prever o **geek_rating** (Bayesian rating do BGG) a partir da base já tratada em `tratamento_dados.ipynb`:

| Modelo | Tipo | Target usado |
|---|---|---|
| Regressão Linear | Regressão | `geek_rating` contínuo |
| Random Forest Regressor | Regressão | `geek_rating` contínuo |
| Gradient Boosting Regressor | Regressão | `geek_rating` contínuo |
| Regressão Logística | Classificação (10 classes) | `geek_rating` discretizado em 10 faixas |

**Por que a Regressão Logística é tratada separadamente?**
Regressão logística é, por natureza, um modelo de **classificação**, não de regressão contínua. Para usá-la na previsão de `geek_rating`, discretizamos o valor contínuo em **10 faixas (bins) de largura igual**, cobrindo o intervalo observado de `geek_rating` no conjunto de treino — ou seja, transformamos o problema em uma classificação multiclasse com classes de 0 a 9 (decis de amplitude, não de frequência). A Logística **não** passou pela reformulação metodológica abaixo (ela mantém `GridSearchCV` + holdout simples), pois o foco do retrabalho foi nos 3 modelos de regressão.

## O que muda nesta versão em relação à anterior

A versão anterior tinha várias limitações metodológicas, listadas e corrigidas abaixo:

| # | Problema na versão anterior | Correção nesta versão |
|---|---|---|
| 1 | `KFold` simples (sem shuffle) em uma variável fortemente assimétrica | `KFold(shuffle=True, random_state=...)` em todos os splits, para misturar bem as classes raras de `geek_rating` alto antes de particionar |
| 2 | Tuning usando apenas **um** conjunto de validação (`train_test_split` único) | **Nested Cross-Validation**: validação cruzada externa para estimar o erro de generalização + validação cruzada interna para escolher hiperparâmetros, dentro de cada fold externo |
| 3 | — | R² mantido como critério de busca (apropriado para regressão; mantemos a métrica, só trocamos a forma de buscar/validar) |
| 4 | Só 20 amostras no espaço de busca, sem deixar claro o método | Mantemos 20 amostras, mas via `RandomizedSearchCV` (que usa `ParameterSampler` internamente) explicitamente documentado, e repetido em cada fold externo |
| 5 | Não havia baseline | Adicionado `DummyRegressor` (estratégia `mean`) como baseline mínimo de comparação |
| 6 | Não havia validação cruzada do modelo final; fluxo era treino → validação → teste único | Fluxo agora é **Nested CV completo**: a métrica reportada é a média ± desvio padrão dos R² nos folds externos (estimativa honesta do erro de generalização). Um modelo final também é refitado no conjunto treino+validação completo (com nova busca de hiperparâmetros) e avaliado no holdout de teste, para obter um modelo "de produção" + artefatos de diagnóstico |
| 7 | Só histogramas de erro, sem resíduos × predição | Adicionado gráfico de resíduos (erro) vs. valor predito para cada modelo de regressão |
| 8 | `feature_importances_` (enviesado em RF/GB, infla variáveis de alta cardinalidade) | Adicionado **Permutation Importance** (model-agnóstico, mede impacto real no R² fora da amostra) e **SHAP** (explica a contribuição marginal de cada feature por amostra, mais robusto a colinearidade) |

> ⚠️ Nested CV é **caro computacionalmente**: para cada fold externo, fazemos uma busca completa de hiperparâmetros (validação cruzada interna). Usamos um orçamento "leve" (5 folds externos × 3 folds internos × 20 amostras de hiperparâmetros) para manter o tempo de execução razoável. Em uma máquina com mais núcleos, aumentar `N_OUTER`, `N_INNER` e `N_ITER` melhora a robustez da estimativa.

## 1. Imports

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold, RandomizedSearchCV, train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

import shap

pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42

# Orçamento do Nested CV (ver observação na introdução sobre custo computacional)
N_OUTER = 5   # folds da validação cruzada externa (estimativa de generalização)
N_INNER = 3   # folds da validação cruzada interna (escolha de hiperparâmetros)
N_ITER = 20   # nº de combinações de hiperparâmetros amostradas (RandomizedSearchCV / ParameterSampler)

## 2. Carregamento dos dados

Carregamos `bgg_dataset_tratado.csv`, gerado pelo notebook `tratamento_dados.ipynb`. A base já está limpa, sem valores nulos, com variáveis categóricas codificadas (one-hot para categorias/mecânicas), publisher e descrição vetorizados. O `geek_rating` (nosso target) permanece na escala original.

In [ ]:
df = pd.read_csv("bgg_dataset_tratado.csv")

print(f"Shape: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
print(f"Valores nulos totais: {df.isna().sum().sum()}")
df.head()

In [ ]:
df["geek_rating"].describe()

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["geek_rating"], bins=50, kde=True)
plt.title("Distribuição de geek_rating")
plt.xlabel("geek_rating")
plt.ylabel("Frequência")
plt.show()

**Observação importante sobre a distribuição:** o `geek_rating` é fortemente assimétrico à direita. A maioria dos jogos se concentra perto do "piso" do Bayesian average do BGG (em torno de 5.5), e poucos jogos atingem ratings altos (>7.5).

Essa assimetria é justamente o motivo pelo qual usamos `KFold(shuffle=True, random_state=RANDOM_STATE)` em todos os splits de validação cruzada deste notebook: sem embaralhar, um `KFold` simples percorre os dados na ordem em que estão no arquivo, o que pode concentrar os poucos jogos de rating alto em apenas um ou dois folds — fazendo com que alguns folds de treino/validação nunca vejam exemplos representativos da cauda direita da distribuição. Isso afeta diretamente a versão de classificação (Regressão Logística): as faixas mais altas terão poucas amostras, o que é esperado e será discutido nos resultados.

## 3. Separação de features e target

- `X`: todas as colunas exceto `geek_rating`.
- `y_reg`: o `geek_rating` contínuo (usado em Regressão Linear, Random Forest e Gradient Boosting).
- O target de classificação (`y_clf`) será derivado de `y_reg` *depois* do split treino/teste, para não vazar informação do teste na definição das faixas (usado apenas na Regressão Logística, seção 11).

In [ ]:
X = df.drop(columns=["geek_rating"])
y_reg = df["geek_rating"]

print(f"X: {X.shape}")
print(f"y_reg: {y_reg.shape}")

## 4. Holdout de teste final

Separamos uma fração dos dados (20%) como **holdout de teste**, que só é usado uma vez, no fim, para avaliar o modelo final de cada algoritmo (seção 8). Esse split usa `shuffle=True` (padrão do `train_test_split`) — importante pela mesma razão explicada na seção 2: a variável é assimétrica, então precisamos garantir que treino e teste tenham representação proporcional de toda a faixa de `geek_rating`.

O conjunto `X_train_full` / `y_train_full_reg` (80% restante) é o que entra no **Nested Cross-Validation** da seção 7: dentro dele, a validação cruzada externa cria seus próprios folds de "treino interno" e "validação", então não precisamos (e não devemos) separar um conjunto de validação fixo aqui — isso é exatamente o problema do item 2 da lista de correções (tuning com um único conjunto de validação) que estamos eliminando.

In [ ]:
X_train_full, X_test, y_train_full_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Treino+Validação (usado no Nested CV): {X_train_full.shape[0]:,} amostras")
print(f"Teste (holdout final):                  {X_test.shape[0]:,} amostras")

## 5. Baseline

Antes de qualquer modelo "de verdade", estabelecemos um **baseline** ingênuo: prever sempre a média do `geek_rating` do conjunto de treino. Qualquer modelo que não supere esse baseline com folga não está agregando valor real. O `DummyRegressor(strategy="mean")` não usa nenhuma feature de `X` — serve só como piso de comparação.

In [ ]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train_full, y_train_full_reg)
y_pred_baseline = baseline.predict(X_test)

rmse_baseline = mean_squared_error(y_test_reg, y_pred_baseline) ** 0.5
mae_baseline = mean_absolute_error(y_test_reg, y_pred_baseline)
r2_baseline = r2_score(y_test_reg, y_pred_baseline)

resultado_baseline = {"modelo": "Baseline (média)", "RMSE": rmse_baseline, "MAE": mae_baseline, "R2": r2_baseline}

print("--- Baseline (prevê sempre a média do treino) ---")
print(f"RMSE: {rmse_baseline:.4f}")
print(f"MAE:  {mae_baseline:.4f}")
print(f"R²:   {r2_baseline:.4f}  (por definição, R² do baseline-média é ~0 no próprio treino; no teste pode ser levemente negativo)")

## 6. Função auxiliar de avaliação (regressão)

Função utilitária para reportar RMSE, MAE e R² de forma consistente entre os modelos.

In [ ]:
def avaliar_regressao(nome_modelo, modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    rmse = mean_squared_error(y_eval, y_pred) ** 0.5
    mae = mean_absolute_error(y_eval, y_pred)
    r2 = r2_score(y_eval, y_pred)

    print(f"--- {nome_modelo} ---")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"R²:   {r2:.4f}")

    return {"modelo": nome_modelo, "RMSE": rmse, "MAE": mae, "R2": r2}

## 7. Nested Cross-Validation — função genérica

Esta é a peça central da reformulação. Para cada modelo, fazemos:

1. **Loop externo** (`outer_cv`, `N_OUTER` folds, `shuffle=True`): divide `X_train_full` em treino-interno / validação-externa. A métrica calculada na validação-externa de cada fold é a nossa estimativa **honesta** de erro de generalização, porque o fold de validação-externa nunca participa da escolha de hiperparâmetros.
2. **Loop interno** (`inner_cv`, `N_INNER` folds, `shuffle=True`), dentro de cada fold externo: usamos `RandomizedSearchCV` (que internamente usa `ParameterSampler` para sortear `N_ITER` combinações do espaço de hiperparâmetros) com `scoring="r2"` para escolher os melhores hiperparâmetros **usando apenas o treino-interno desse fold**.
3. O melhor estimador da busca interna é então avaliado no fold de validação-externa (dados nunca vistos pela busca de hiperparâmetros).

Ao final, temos `N_OUTER` valores de R² (e RMSE/MAE) — reportamos a **média ± desvio padrão**, que é uma estimativa muito mais confiável do desempenho esperado em dados novos do que um único número de um único split de validação.

> Note a diferença para o fluxo antigo (treino → validação → teste): ali, o mesmo conjunto de validação era usado repetidamente para escolher hiperparâmetros, o que tende a gerar uma estimativa **otimista** (overfitting nos hiperparâmetros em relação àquele conjunto específico). O Nested CV resolve isso ao nunca deixar o conjunto de avaliação tocar a escolha de hiperparâmetros.

In [ ]:
def nested_cv_regressao(nome_modelo, pipeline, param_distributions, X, y,
                          n_outer=N_OUTER, n_inner=N_INNER, n_iter=N_ITER,
                          random_state=RANDOM_STATE):
    """Executa Nested Cross-Validation para um pipeline de regressão.

    Retorna um dicionário com as métricas agregadas (média e desvio padrão
    dos folds externos) e a lista de melhores hiperparâmetros encontrados
    em cada fold externo (útil para inspecionar estabilidade da busca).
    """
    outer_cv = KFold(n_splits=n_outer, shuffle=True, random_state=random_state)

    r2_scores, rmse_scores, mae_scores = [], [], []
    melhores_params_por_fold = []

    t0 = time.time()
    for fold_i, (train_idx, val_idx) in enumerate(outer_cv.split(X), start=1):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        inner_cv = KFold(n_splits=n_inner, shuffle=True, random_state=random_state)

        busca = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=param_distributions,
            n_iter=n_iter,
            cv=inner_cv,
            scoring="r2",
            random_state=random_state,
            n_jobs=-1,
        )
        busca.fit(X_tr, y_tr)

        melhor_modelo = busca.best_estimator_
        y_pred_val = melhor_modelo.predict(X_val)

        r2_fold = r2_score(y_val, y_pred_val)
        rmse_fold = mean_squared_error(y_val, y_pred_val) ** 0.5
        mae_fold = mean_absolute_error(y_val, y_pred_val)

        r2_scores.append(r2_fold)
        rmse_scores.append(rmse_fold)
        mae_scores.append(mae_fold)
        melhores_params_por_fold.append(busca.best_params_)

        print(f"  [{nome_modelo}] Fold externo {fold_i}/{n_outer} — "
              f"R²={r2_fold:.4f}  RMSE={rmse_fold:.4f}  MAE={mae_fold:.4f}  "
              f"melhores_params={busca.best_params_}")

    tempo_total = time.time() - t0
    print(f"  [{nome_modelo}] Nested CV concluído em {tempo_total:.1f}s")

    resultado = {
        "modelo": nome_modelo,
        "R2_mean": np.mean(r2_scores),
        "R2_std": np.std(r2_scores),
        "RMSE_mean": np.mean(rmse_scores),
        "RMSE_std": np.std(rmse_scores),
        "MAE_mean": np.mean(mae_scores),
        "MAE_std": np.std(mae_scores),
        "r2_scores_por_fold": r2_scores,
        "rmse_scores_por_fold": rmse_scores,
        "mae_scores_por_fold": mae_scores,
        "melhores_params_por_fold": melhores_params_por_fold,
        "tempo_segundos": tempo_total,
    }
    return resultado

## 8. Modelo 1 — Regressão Linear

Modelo baseline "estatístico". O espaço de hiperparâmetros é pequeno, mas ainda assim passamos pelo Nested CV para manter a metodologia consistente entre os modelos (e porque `RandomizedSearchCV` com um espaço pequeno simplesmente amostra todas as combinações possíveis, sem problema).

In [ ]:
pipeline_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

param_dist_linear = {
    "model__fit_intercept": [True, False],
    "model__positive": [True, False],
}

resultado_nested_linear = nested_cv_regressao(
    "Regressão Linear", pipeline_linear, param_dist_linear, X_train_full, y_train_full_reg
)

print()
print(f"R² (Nested CV): {resultado_nested_linear['R2_mean']:.4f} ± {resultado_nested_linear['R2_std']:.4f}")

## 9. Modelo 2 — Random Forest Regressor

Random Forest costuma capturar bem relações não-lineares entre as features (ex: número de votos, complexidade, categorias) e o `geek_rating`.

> 💡 O espaço de busca foi dimensionado para manter o Nested CV viável em tempo razoável (5 folds externos × busca interna de 20 combinações × 3 folds internos = até 305 treinamentos de Random Forest). Se tiver mais capacidade computacional, vale ampliar o espaço de `param_dist_rf` e/ou aumentar `N_OUTER`/`N_INNER`/`N_ITER` na seção 1.

In [ ]:
pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1))
])

param_dist_rf = {
    "model__n_estimators": [100, 150, 200],
    "model__max_depth": [6, 8, 12, None],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__max_features": ["sqrt", 0.5, None],
}

resultado_nested_rf = nested_cv_regressao(
    "Random Forest", pipeline_rf, param_dist_rf, X_train_full, y_train_full_reg
)

print()
print(f"R² (Nested CV): {resultado_nested_rf['R2_mean']:.4f} ± {resultado_nested_rf['R2_std']:.4f}")

## 10. Modelo 3 — Gradient Boosting Regressor

Gradient Boosting ajusta árvores sequencialmente, cada uma corrigindo os erros da anterior. Tende a ter desempenho competitivo, ao custo de mais tempo de treino — o que pesa ainda mais dentro de um Nested CV.

In [ ]:
pipeline_gb = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingRegressor(random_state=RANDOM_STATE))
])

param_dist_gb = {
    "model__n_estimators": [100, 150, 200],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__max_depth": [2, 3, 4],
    "model__subsample": [0.8, 1.0],
}

resultado_nested_gb = nested_cv_regressao(
    "Gradient Boosting", pipeline_gb, param_dist_gb, X_train_full, y_train_full_reg
)

print()
print(f"R² (Nested CV): {resultado_nested_gb['R2_mean']:.4f} ± {resultado_nested_gb['R2_std']:.4f}")

## 11. Modelos finais — refit completo e avaliação no holdout de teste

O Nested CV da seção anterior nos dá uma **estimativa confiável do erro de generalização** (R² médio ± desvio entre folds), mas não produz, por si só, um único modelo "final" para usar em produção — cada fold externo teve seus próprios hiperparâmetros.

Para obter um modelo final único:

1. Fazemos uma última busca de hiperparâmetros (`RandomizedSearchCV`, mesmo espaço e `N_ITER`, com validação cruzada em `X_train_full` inteiro) para escolher os hiperparâmetros finais.
2. Treinamos esse modelo com os hiperparâmetros escolhidos em **todo** `X_train_full`.
3. Avaliamos no `X_test` (holdout que nunca foi tocado até agora).

Essa avaliação no holdout deve ficar **na mesma faixa** do R² médio do Nested CV (seção anterior) — se for muito diferente, é sinal de instabilidade. O número do Nested CV é a estimativa que reportamos como "desempenho esperado do modelo"; o holdout serve de checagem adicional e gera o modelo final usado nos diagnósticos (resíduos, importâncias) a seguir.

In [ ]:
def treinar_modelo_final(nome_modelo, pipeline, param_distributions, X_tr_full, y_tr_full, X_te, y_te,
                           n_inner=N_INNER, n_iter=N_ITER, random_state=RANDOM_STATE):
    inner_cv = KFold(n_splits=n_inner, shuffle=True, random_state=random_state)
    busca_final = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=n_iter,
        cv=inner_cv,
        scoring="r2",
        random_state=random_state,
        n_jobs=-1,
    )
    busca_final.fit(X_tr_full, y_tr_full)

    modelo_final = busca_final.best_estimator_
    resultado_holdout = avaliar_regressao(nome_modelo, modelo_final, X_te, y_te)

    print(f"Melhores hiperparâmetros (refit final): {busca_final.best_params_}")
    return modelo_final, resultado_holdout, busca_final.best_params_

In [ ]:
modelo_linear, resultado_linear, params_linear = treinar_modelo_final(
    "Regressão Linear", pipeline_linear, param_dist_linear, X_train_full, y_train_full_reg, X_test, y_test_reg
)

In [ ]:
modelo_rf, resultado_rf, params_rf = treinar_modelo_final(
    "Random Forest", pipeline_rf, param_dist_rf, X_train_full, y_train_full_reg, X_test, y_test_reg
)

In [ ]:
modelo_gb, resultado_gb, params_gb = treinar_modelo_final(
    "Gradient Boosting", pipeline_gb, param_dist_gb, X_train_full, y_train_full_reg, X_test, y_test_reg
)

## 12. Comparação dos modelos — baseline, Nested CV e holdout final

In [ ]:
tabela_nested = pd.DataFrame([
    {"modelo": r["modelo"], "R2_mean_nestedCV": r["R2_mean"], "R2_std_nestedCV": r["R2_std"],
     "RMSE_mean_nestedCV": r["RMSE_mean"], "MAE_mean_nestedCV": r["MAE_mean"]}
    for r in [resultado_nested_linear, resultado_nested_rf, resultado_nested_gb]
]).set_index("modelo")

tabela_holdout = pd.DataFrame([resultado_baseline, resultado_linear, resultado_rf, resultado_gb]).set_index("modelo")
tabela_holdout = tabela_holdout.rename(columns={"R2": "R2_holdout", "RMSE": "RMSE_holdout", "MAE": "MAE_holdout"})

tabela_comparativa = tabela_holdout.join(tabela_nested, how="left")
tabela_comparativa = tabela_comparativa.sort_values("R2_holdout", ascending=False)
print("Comparação completa — Baseline, Nested CV (treino+validação) e Holdout (teste final)")
tabela_comparativa

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

modelos_plot = tabela_comparativa.index
x_pos = np.arange(len(modelos_plot))

axes[0].bar(x_pos, tabela_comparativa["R2_holdout"], color="#4C72B0", label="R² holdout (teste)")
axes[0].errorbar(x_pos, tabela_comparativa["R2_mean_nestedCV"],
                  yerr=tabela_comparativa["R2_std_nestedCV"],
                  fmt="o", color="black", label="R² Nested CV (média ± desvio)")
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(modelos_plot, rotation=30, ha="right")
axes[0].set_title("R²: holdout vs Nested CV")
axes[0].axhline(0, color="grey", linewidth=0.8)
axes[0].legend()

axes[1].bar(x_pos, tabela_comparativa["RMSE_holdout"], color="#DD8452")
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(modelos_plot, rotation=30, ha="right")
axes[1].set_title("RMSE (holdout)")

plt.tight_layout()
plt.show()

Se as barras de R² do holdout estiverem dentro (ou muito próximas) do intervalo média ± desvio do Nested CV, isso é um bom sinal de que a estimativa de generalização é estável e não dependeu de sorte no split de teste.

## 13. Análise de resíduos

Histograma de erro sozinho não revela **padrões sistemáticos** do modelo (ex: o modelo erra mais para jogos com `geek_rating` alto? O erro tem variância constante ao longo da faixa de predição — homocedasticidade — ou cresce/diminui — heterocedasticidade?). Por isso, além do histograma, plotamos **resíduo (erro) × valor predito** para cada modelo de regressão.

- Resíduo = `y_real - y_predito`.
- Um modelo bem ajustado deve ter resíduos espalhados aleatoriamente em torno de zero, sem padrão (sem "funil", sem curvatura, sem tendência).
- Padrões visíveis (ex: resíduos sistematicamente positivos em uma faixa e negativos em outra) indicam que o modelo está deixando estrutura não capturada nos dados.

In [ ]:
def plot_residuos(nome_modelo, modelo, X_eval, y_eval, ax_resid, ax_hist):
    y_pred = modelo.predict(X_eval)
    residuos = y_eval.values - y_pred

    ax_resid.scatter(y_pred, residuos, alpha=0.35, s=12, color="#4C72B0")
    ax_resid.axhline(0, color="red", linestyle="--", linewidth=1)
    ax_resid.set_xlabel("Valor predito")
    ax_resid.set_ylabel("Resíduo (real - predito)")
    ax_resid.set_title(f"Resíduos × Predição — {nome_modelo}")

    sns.histplot(residuos, bins=40, kde=True, ax=ax_hist, color="#55A868")
    ax_hist.axvline(0, color="red", linestyle="--", linewidth=1)
    ax_hist.set_xlabel("Resíduo")
    ax_hist.set_title(f"Distribuição dos resíduos — {nome_modelo}")

    return residuos

In [ ]:
modelos_finais = [
    ("Regressão Linear", modelo_linear),
    ("Random Forest", modelo_rf),
    ("Gradient Boosting", modelo_gb),
]

fig, axes = plt.subplots(len(modelos_finais), 2, figsize=(12, 4 * len(modelos_finais)))

residuos_por_modelo = {}
for i, (nome, modelo) in enumerate(modelos_finais):
    residuos = plot_residuos(nome, modelo, X_test, y_test_reg, axes[i, 0], axes[i, 1])
    residuos_por_modelo[nome] = residuos

plt.tight_layout()
plt.show()

**Como interpretar:** como `geek_rating` é assimétrico à direita (poucos jogos com rating muito alto), é esperado ver maior dispersão de resíduos justamente nas predições mais altas — há poucos exemplos de treino nessa faixa, então o modelo tem mais incerteza ali. Um padrão de "funil" crescente (resíduos cada vez mais dispersos conforme o valor predito aumenta) é um indício consistente dessa escassez de dados na cauda direita, mais do que necessariamente um erro de modelagem.

## 14. Feature importance — Permutation Importance e SHAP

`feature_importances_` (usado na versão anterior) mede, para árvores, o quanto cada feature reduziu a impureza dos nós em que foi usada para split. Esse critério tem dois problemas conhecidos:

- **Viés para features de alta cardinalidade** (ex: variáveis numéricas contínuas ou one-hot muito esparsas tendem a aparecer artificialmente "importantes" só porque oferecem mais pontos de corte possíveis).
- É calculado **no próprio treino**, então não diz nada sobre o quanto a feature realmente ajuda a generalizar para dados novos.

Por isso, usamos duas alternativas mais confiáveis, calculadas no conjunto de **teste** (holdout):

1. **Permutation Importance**: embaralha os valores de uma feature por vez (mantendo as demais intactas) e mede o quanto o R² do modelo cai no teste. Quanto maior a queda, mais importante a feature é *para a generalização* — é model-agnóstico e não sofre do viés de cardinalidade do `feature_importances_`.
2. **SHAP (SHapley Additive exPlanations)**: distribui a predição de cada amostra entre as features, com base em teoria de jogos cooperativos (valores de Shapley), respeitando interações e correlações entre features de forma mais rigorosa que a permutação simples. Usamos `TreeExplainer` (exato e eficiente para modelos baseados em árvore como RF e GB).

> A Regressão Linear não sofre do mesmo viés — seus coeficientes (após padronização das features) já são diretamente interpretáveis como importância, então não repetimos Permutation/SHAP para ela (mostramos os coeficientes na seção 14.3).

### 14.1 Permutation Importance (Random Forest e Gradient Boosting)

In [ ]:
def calcular_permutation_importance(nome_modelo, modelo, X_eval, y_eval, n_repeats=10):
    resultado_perm = permutation_importance(
        modelo, X_eval, y_eval,
        scoring="r2", n_repeats=n_repeats, random_state=RANDOM_STATE, n_jobs=-1
    )
    importancias = pd.Series(resultado_perm.importances_mean, index=X_eval.columns)
    desvios = pd.Series(resultado_perm.importances_std, index=X_eval.columns)
    return importancias, desvios

In [ ]:
perm_imp_rf, perm_std_rf = calcular_permutation_importance("Random Forest", modelo_rf, X_test, y_test_reg)
perm_imp_gb, perm_std_gb = calcular_permutation_importance("Gradient Boosting", modelo_gb, X_test, y_test_reg)

top_n = 15
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

top_rf = perm_imp_rf.sort_values(ascending=False).head(top_n)
axes[0].barh(top_rf.index[::-1], top_rf.values[::-1],
             xerr=perm_std_rf.loc[top_rf.index][::-1], color="#4C72B0")
axes[0].set_title(f"Permutation Importance — Random Forest (top {top_n})")
axes[0].set_xlabel("Queda média de R² ao embaralhar a feature")

top_gb = perm_imp_gb.sort_values(ascending=False).head(top_n)
axes[1].barh(top_gb.index[::-1], top_gb.values[::-1],
             xerr=perm_std_gb.loc[top_gb.index][::-1], color="#DD8452")
axes[1].set_title(f"Permutation Importance — Gradient Boosting (top {top_n})")
axes[1].set_xlabel("Queda média de R² ao embaralhar a feature")

plt.tight_layout()
plt.show()

### 14.2 SHAP (Random Forest e Gradient Boosting)

Para manter o tempo de execução razoável (o dataset tem mais de 150 features), calculamos SHAP em uma **amostra** do conjunto de teste, em vez do teste completo. Isso é uma prática padrão: o objetivo do SHAP aqui é entender o padrão geral de importância e o sentido (positivo/negativo) do efeito de cada feature, não auditar cada predição individual — uma amostra de algumas centenas de linhas já estima esse padrão de forma estável.

Como o modelo é um `Pipeline` (`StandardScaler` + modelo de árvore), aplicamos o `TreeExplainer` diretamente sobre o **modelo de árvore extraído do pipeline**, usando os dados já transformados pelo `StandardScaler` — assim o SHAP enxerga exatamente os mesmos valores de entrada que o modelo de árvore usa internamente.

In [ ]:
N_AMOSTRA_SHAP = 500
amostra_shap = X_test.sample(n=min(N_AMOSTRA_SHAP, len(X_test)), random_state=RANDOM_STATE)

def calcular_shap_values(pipeline, amostra):
    scaler = pipeline.named_steps["scaler"]
    modelo_arvore = pipeline.named_steps["model"]
    amostra_escalada = pd.DataFrame(
        scaler.transform(amostra), columns=amostra.columns, index=amostra.index
    )
    explainer = shap.TreeExplainer(modelo_arvore)
    shap_values = explainer.shap_values(amostra_escalada)
    return shap_values, amostra_escalada

In [ ]:
shap_values_rf, amostra_escalada_rf = calcular_shap_values(modelo_rf, amostra_shap)

plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values_rf, amostra_escalada_rf, max_display=15, show=False)
plt.title("SHAP summary — Random Forest")
plt.tight_layout()
plt.show()

In [ ]:
shap_values_gb, amostra_escalada_gb = calcular_shap_values(modelo_gb, amostra_shap)

plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values_gb, amostra_escalada_gb, max_display=15, show=False)
plt.title("SHAP summary — Gradient Boosting")
plt.tight_layout()
plt.show()

**Como ler o gráfico SHAP summary:** cada ponto é uma amostra do conjunto de teste. A posição horizontal mostra o impacto daquela feature na predição daquela amostra específica (negativo = empurra a predição para baixo, positivo = empurra para cima). A cor mostra o valor da própria feature (vermelho = valor alto, azul = valor baixo) — isso revela não só *quão* importante a feature é, mas também *em que direção* ela afeta o `geek_rating`, algo que `feature_importances_` nunca conseguiria mostrar.

### 14.3 Comparação Random Forest vs Gradient Boosting (Permutation Importance) e coeficientes da Regressão Linear

In [ ]:
comparacao_perm = pd.DataFrame({
    "Random Forest": perm_imp_rf,
    "Gradient Boosting": perm_imp_gb,
}).sort_values("Random Forest", ascending=False).head(15)

comparacao_perm.plot(kind="barh", figsize=(8, 6))
plt.title("Permutation Importance — RF vs GB (top 15 por RF)")
plt.xlabel("Queda média de R²")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
coef_linear = pd.Series(
    modelo_linear.named_steps["model"].coef_, index=X_train_full.columns
)
top_coef = coef_linear.reindex(coef_linear.abs().sort_values(ascending=False).head(15).index)

plt.figure(figsize=(8, 5))
top_coef.sort_values().plot(kind="barh", color=top_coef.sort_values().apply(lambda v: "#C44E52" if v < 0 else "#4C72B0"))
plt.title("Top 15 coeficientes (|valor|) — Regressão Linear (features padronizadas)")
plt.xlabel("Coeficiente (efeito por desvio-padrão da feature)")
plt.axvline(0, color="grey", linewidth=0.8)
plt.tight_layout()
plt.show()

## 15. Modelo 4 — Regressão Logística (classificação em 10 faixas)

Esta seção foi **mantida como na versão original** (conforme decisão de foco da reformulação): `GridSearchCV` com holdout simples de treino/teste, sem Nested CV, sem SHAP. O objetivo dela é apenas oferecer uma visão complementar de classificação, não é o foco da reformulação metodológica.

### 15.1 Discretização do target

Criamos 10 faixas de **largura igual** (bins lineares, `np.linspace`) cobrindo o intervalo `[min, max]` de `geek_rating` observado **no treino** (`X_train_full` / `y_train_full_reg`, a mesma partição treino+validação usada no Nested CV das seções anteriores). As extremidades dos bins são abertas para `-inf`/`+inf`, garantindo que qualquer valor do teste (mesmo fora do range do treino) seja classificado em alguma faixa, sem gerar `NaN`.

In [ ]:
N_BINS = 10

bin_edges = np.linspace(y_train_full_reg.min(), y_train_full_reg.max(), N_BINS + 1)
bin_labels_desc = [f"{bin_edges[i]:.3f} – {bin_edges[i+1]:.3f}" for i in range(N_BINS)]

bin_edges_open = bin_edges.copy()
bin_edges_open[0] = -np.inf
bin_edges_open[-1] = np.inf

y_train_clf = pd.cut(y_train_full_reg, bins=bin_edges_open, labels=False, include_lowest=True)
y_test_clf = pd.cut(y_test_reg, bins=bin_edges_open, labels=False, include_lowest=True)

print("Faixas de geek_rating (definidas a partir do treino):")
for i, lbl in enumerate(bin_labels_desc):
    print(f"  Classe {i}: {lbl}")

In [ ]:
dist_treino = y_train_clf.value_counts().sort_index()
dist_teste = y_test_clf.value_counts().sort_index()

dist_df = pd.DataFrame({"treino": dist_treino, "teste": dist_teste}).fillna(0).astype(int)
print(dist_df)

plt.figure(figsize=(9, 4))
dist_df.plot(kind="bar", ax=plt.gca())
plt.title("Distribuição das classes (faixas de geek_rating)")
plt.xlabel("Classe (faixa)")
plt.ylabel("Quantidade de jogos")
plt.tight_layout()
plt.show()

⚠️ **Forte desbalanceamento de classes.** As classes 0 e 1 concentram a grande maioria dos jogos; as classes mais altas (7, 8, 9) têm poucas dezenas de exemplos. Por isso:

- Usamos `class_weight="balanced"` na Regressão Logística, para compensar o desbalanceamento durante o treino.
- Avaliamos com **F1-score ponderado (weighted)** como métrica principal do `GridSearchCV` (mais robusta a desbalanceamento que a acurácia simples), além de reportar accuracy e o relatório completo por classe.

### 15.2 Treinamento

In [ ]:
pipeline_logistic = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ))
])

param_grid_logistic = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__solver": ["lbfgs"],
}

t0 = time.time()
from sklearn.model_selection import GridSearchCV
grid_logistic = GridSearchCV(
    estimator=pipeline_logistic,
    param_grid=param_grid_logistic,
    cv=3,
    scoring="f1_weighted",
    n_jobs=-1,
)
grid_logistic.fit(X_train_full, y_train_clf)

print(f"Tempo de treinamento: {time.time() - t0:.1f}s")
print(f"Melhores hiperparâmetros: {grid_logistic.best_params_}")
print(f"Melhor F1 ponderado (validação cruzada): {grid_logistic.best_score_:.4f}")

In [ ]:
modelo_logistic = grid_logistic.best_estimator_
y_pred_clf = modelo_logistic.predict(X_test)

acc = accuracy_score(y_test_clf, y_pred_clf)
f1_weighted = f1_score(y_test_clf, y_pred_clf, average="weighted")
precision_weighted = precision_score(y_test_clf, y_pred_clf, average="weighted", zero_division=0)
recall_weighted = recall_score(y_test_clf, y_pred_clf, average="weighted", zero_division=0)

print("--- Regressão Logística (10 classes) ---")
print(f"Accuracy:            {acc:.4f}")
print(f"F1 (weighted):        {f1_weighted:.4f}")
print(f"Precision (weighted): {precision_weighted:.4f}")
print(f"Recall (weighted):    {recall_weighted:.4f}")
print()
print(classification_report(y_test_clf, y_pred_clf, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test_clf, y_pred_clf, labels=range(N_BINS))

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=range(N_BINS), yticklabels=range(N_BINS))
plt.title("Matriz de confusão — Regressão Logística (10 faixas)")
plt.xlabel("Classe prevista")
plt.ylabel("Classe real")
plt.tight_layout()
plt.show()

In [ ]:
resultado_logistic = {
    "modelo": "Regressão Logística (10 classes)",
    "Accuracy": acc,
    "F1_weighted": f1_weighted,
    "Precision_weighted": precision_weighted,
    "Recall_weighted": recall_weighted,
}

tabela_classificacao = pd.DataFrame([resultado_logistic]).set_index("modelo")
print("Resultado — Regressão Logística (target discretizado em 10 faixas)")
tabela_classificacao

## 16. Resumo dos hiperparâmetros finais e estabilidade entre folds

Além dos hiperparâmetros do modelo final (refit em `X_train_full` completo), mostramos também os hiperparâmetros escolhidos em **cada fold externo** do Nested CV. Se a busca interna escolher combinações muito diferentes a cada fold, isso é um sinal de que o espaço de hiperparâmetros tem vários "ótimos" parecidos entre si (instabilidade de seleção, mas não necessariamente de desempenho) — vale olhar tanto a tabela de R² por fold (seções 9-10) quanto esta tabela.

In [ ]:
resumo_hiperparametros = pd.DataFrame([
    {"modelo": "Regressão Linear", "melhores_hiperparametros_final": params_linear,
     "R2_holdout": resultado_linear["R2"], "R2_mean_nestedCV": resultado_nested_linear["R2_mean"]},
    {"modelo": "Random Forest", "melhores_hiperparametros_final": params_rf,
     "R2_holdout": resultado_rf["R2"], "R2_mean_nestedCV": resultado_nested_rf["R2_mean"]},
    {"modelo": "Gradient Boosting", "melhores_hiperparametros_final": params_gb,
     "R2_holdout": resultado_gb["R2"], "R2_mean_nestedCV": resultado_nested_gb["R2_mean"]},
    {"modelo": "Regressão Logística", "melhores_hiperparametros_final": grid_logistic.best_params_,
     "R2_holdout": np.nan, "R2_mean_nestedCV": np.nan},
])
resumo_hiperparametros

In [ ]:
print("Hiperparâmetros escolhidos em cada fold externo do Nested CV:\n")
for resultado in [resultado_nested_linear, resultado_nested_rf, resultado_nested_gb]:
    print(f"--- {resultado['modelo']} ---")
    for i, params in enumerate(resultado["melhores_params_por_fold"], start=1):
        print(f"  Fold {i}: {params}")
    print()

## 17. Conclusões

- **Baseline:** o `DummyRegressor` (prevê sempre a média) estabelece o piso de comparação — todos os modelos de regressão devem (e devem-se reportar caso não) superá-lo com folga em R².
- **Nested CV vs. holdout único:** a métrica de generalização reportada para Linear/RF/GB agora é a média ± desvio padrão de R² entre `N_OUTER` folds externos, cada um com sua própria busca de hiperparâmetros (`RandomizedSearchCV`, `N_ITER` amostras, `N_INNER` folds internos). Isso é mais robusto do que o fluxo anterior (treino → validação única → teste), que tendia a gerar uma estimativa otimista por reutilizar o mesmo conjunto de validação na escolha de hiperparâmetros.
- **`KFold(shuffle=True)`** em todos os splits evita que a forte assimetria de `geek_rating` concentre os jogos de rating alto em poucos folds.
- **Resíduos:** o gráfico de resíduo × predição complementa o histograma de erros e ajuda a identificar heterocedasticidade — esperamos (e observamos) maior dispersão nas predições mais altas, refletindo a escassez de exemplos na cauda direita da distribuição do `geek_rating`.
- **Importância de features:** Permutation Importance e SHAP substituem o `feature_importances_` nativo de RF/GB, que tende a inflar artificialmente a importância de variáveis numéricas/esparsas de alta cardinalidade. As duas técnicas concordam em geral sobre quais features mais contribuem para a predição, e o SHAP adicionalmente revela a direção do efeito.
- A Regressão Logística resolve uma tarefa diferente (classificar em qual de 10 faixas o jogo cai) e foi mantida com a metodologia original (`GridSearchCV` + holdout), por estar fora do escopo da reformulação solicitada.
- O forte desbalanceamento de classes em `geek_rating` (poucos jogos com rating muito alto) afeta principalmente a Regressão Logística nas faixas superiores — isso é esperado e reflete a própria distribuição do Bayesian rating do BGG, não um erro no pipeline.